In [4]:
import json
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
import os
os.environ["DEEPSEEK_API_KEY"] = "sk-ae6d423f13b7403fa8ba16ddb4acc297"

In [5]:
from langchain.chat_models import init_chat_model

llm=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

from langchain_huggingface import HuggingFaceEmbeddings
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 
embeddings_model=HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)

In [6]:
def normalize(vec):
    """向量归一化，用于余弦相似度"""
    return vec / (np.linalg.norm(vec) + 1e-8)

In [7]:
# =========================
# 1. 读取 md 并按段落切分
def load_md_and_split(md_path):
    """
    输入：md 文件路径
    输出：段落列表（每段是一段连续文本）
    """
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() and p[0] != '#']
    return paragraphs

In [8]:
# =========================
# 2. LLM：段落级抽实体 + 关系
def extract_entities_and_relations(llm, paragraph):
    """
    输入：一段教材文本
    输出：
    {
        "entities": ["流水", "岩石", ...],
        "relations": [
            {"head": "流水", "relation": "作用于", "tail": "岩石"},
            ...
        ]
    }
    """

    prompt = ChatPromptTemplate.from_template(
        """
        你在做高中地理知识图谱构建。
        从下面这段话中抽取：
        1）所有核心地理实体（名词或概念），不遗漏、不重复、不多余
        2）实体之间的明确关系。注意所用实体必须属于1）中抽取的
        3）实体应当是规范的学科名词，而不是任意名词都可以
        
        只返回 JSON，只输出 JSON，不要任何解释、不要 markdown、不要多余文本。
        格式如下：
        {{
          "entities": [...],
          "relations": [
            {{"origin": "...", "relation": "...", "target": "..."}}
          ]
        }}
        
        文本：
        {paragraph}
        """
    )

    messages = prompt.format_messages(paragraph=paragraph)
    resp = llm.invoke(messages)
    text = resp.content.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"LLM 未返回 JSON：\n{text}")
        
    return json.loads(text[start:end+1])

In [9]:
def init_graph(embeddings_model):
    vectorstore = FAISS.from_texts(
        texts=["__init__"],   # 占位，不能为空
        embedding=embeddings_model
    )

    return {
        "nodes": {},
        "edges": [],
        "vectorstore": vectorstore
    }


In [10]:
# 让LLM判断当前实体是否 在图中 已存在 可以合并的词
def llm_merge_decision(llm, query, candidates):
    """
    query: 当前实体名
    candidates: embedding召回的候选词列表（字符串）
    返回：第一个被 LLM 判定为同一概念的词，否则 None
    """
    prompt = ChatPromptTemplate.from_template("""
        严格判断下面词a是否和词列表B中的某个词在地理意义上是严格的【同义词】。
        特别注意【包含、上下位、并列关系不属于同义词】，只有地理学中语义范围大小完全一样、没有歧义才可以。
        如果存在同义词，输出这个同义词b。
        如果没有同义词，输出no
        
        不要解释。不要输出任何前后缀、空格等。
        词：{a}
        词列表：{B}
    """)

    messages = prompt.format_messages(a=query,B=candidates)
    resp = llm.invoke(messages)
    resp = resp.content.strip()
    return resp

In [11]:
# =========================
# 4. 实体规范化（FAISS 相似度合并）
def get_or_create_entity(graph, entity_name, entity_embedding, sim_threshold=0.85,k=5):
    """
    用 LangChain FAISS 做知识点的相似度检索
    若命中已有实体则合并，否则新建
    输入：
        entity_name: 当前段落抽取的实体名
        entity_embedding: 该实体的向量（np.ndarray）
    输出：
        canon_name: 规范化后的实体名
    """

    vectorstore = graph["vectorstore"]

    # 从已有词里检索当前词的topk相似词 
    if len(graph["nodes"]) > 0 and 1==0:
        docs_and_scores = vectorstore.similarity_search_with_score_by_vector(
        entity_embedding.tolist(),
        k=k
    )
        if docs_and_scores:
            # 按embedding相似度判断是否重合（不靠谱）
            # doc,distance= docs_and_scores[0]
            # print('相似词：',entity_name,doc.page_content,distance)
            
            # if distance <= 1-sim_threshold:
            #     canon_name = doc.page_content
            #     graph["nodes"][canon_name]["aliases"].add(entity_name)
            #     print('‼过相似',entity_name,canon_name,'distance=',distance)
            #     return canon_name

            # 按llm判断是否相似
            candidates=[doc.page_content for doc, _ in docs_and_scores]
            candidates=" ".join(candidates)
            print('topk：',entity_name,'|',candidates)
            canon=llm_merge_decision(llm, query=entity_name, candidates=candidates)
            if canon!='no' and canon in graph["nodes"]:
                graph["nodes"][canon]["aliases"].add(entity_name)
                print('‼可合并',entity_name,canon)
                return canon

    # 没匹配上：新实体
    if entity_name in graph["nodes"]:
        print('existed',entity_name)
        return entity_name
    canon = entity_name
    graph["nodes"][canon] = {
        "aliases": {entity_name},
        "embedding": entity_embedding
    }

    vectorstore.add_texts(
        texts=[canon],
        metadatas=[{"canon": canon}]
    )

    return canon


In [12]:
# =========================
# 5. 段落级入图（实体 + 关系）
# =========================

def process_paragraph(graph, llm, embedder, paragraph,threshold,k):
    """
    对一个段落：
    - 抽实体 + 关系
    - 实体与已有图做相似度合并
    - 关系中的 head / tail 替换为 canon_name
    - 写入图
    """

    result = extract_entities_and_relations(llm, paragraph)

    # 本段实体名 -> canon_name 映射
    local_entity_map = {}

    # 先处理实体
    for name in result["entities"]:
        emb = embedder.embed_query(name)
        canon = get_or_create_entity(graph, name, np.array(emb),threshold,k)
        local_entity_map[name] = canon

    # 再处理关系（统一用 canon_name）
    for rel in result["relations"]:
        head = local_entity_map.get(rel["origin"])
        tail = local_entity_map.get(rel["target"])
        if head and tail:
            graph["edges"].append({
                "origin": head,
                "relation": rel["relation"],
                "target": tail
            })

In [13]:
# =========================
# 6. 总流程
# =========================

def build_knowledge_graph(md_path, llm, embeddings_model,threshold,k):
    """
    输入：教材 md
    输出：完整知识图谱 json
    """

    paragraphs = load_md_and_split(md_path)
    graph = init_graph(embeddings_model)
    print('一共{}段'.format(len(paragraphs)))
    i=0
    for para in paragraphs:
        i+=1
        print('第{}段'.format(i))
        process_paragraph(graph, llm, embeddings_model, para,threshold,k)

    # embedding 不需要输出
    for node in graph["nodes"].values():
        node.pop("embedding")

    return graph

In [14]:
md_path='./data/cleaned_湘教版地理必修一Chap2.md'
load_md_and_split(md_path)

['地球表面的形态称为地貌。地貌是地球演化的结果，处于不断的发展变化之中。地貌类型多样，规模大小不等，形态特点各异。大自然用流水去冲刷，用山洪去切割，用波浪去拍击，用冰川去凿琢……大自然拥有奇妙技法和独特审美，千姿百态的地表形态是它的习作，印证着岁月沧桑。大自然，大创造，天地之中有大美！我们欣赏锦绣山河的多彩容颜，我们更赞叹大千世界的鬼斧神工。',
 '位于四川南充嘉陵江边的青居镇，镇的南北各建有一个码头，北边的叫上码头，南边的叫下码头。青居人家到曲水赶场、走亲戚，去时从上码头乘船是顺水，回来乘船在下码头上岸也是顺水，形成了可能是我国河道里唯一的“来也顺水，去也顺水”的奇特航程。旧时，纤夫早上从下码头出发，傍晚投宿上码头，依然住进头天晚上的客栈，素有“行船走一天，步行一袋烟”之说。',
 '河流流向',
 '图2-1\u3000嘉陵江青居镇附近遥感影像',
 '1.嘉陵江在这一段为什么会形成这样的形状呢？',
 '2.想一想，流水能塑造出哪些地表形态？试举例说明。',
 '流水是地表常见的外力作用形式。流水塑造的地貌，称为流水地貌。一般来说，流水地貌可分为流水侵蚀地貌和流水堆积地貌。',
 '在湿润或半湿润山区，流水侵蚀切割地面形成峡谷，河谷横断面大多呈 V 字形。流水下切作用强烈的地区，往往形成深邃的峡谷。',
 '河谷中枯水期出露、洪水期淹没的部分称为河漫滩。在河谷两侧常分布有洪水不能淹没的阶梯状地形，称为河流阶地。河流阶地地面平坦，组成物质颗粒较细，土质较为肥沃。',
 '图 2-2\u3000河谷横剖面结构示意 1.河床2.河漫滩3.阶地 枯水位---洪水位',
 '虎跳峡位于玉龙雪山与哈巴雪山之间，两岸高山夹峙，峭壁耸立，山岭高出江面达 3 000 米以上，中间江面宽仅 30~60 米，湍流激荡，山轰谷鸣，气势非凡。',
 '为什么人们常选择河流阶地作为居住和耕作的场所？',
 '平原地区蜿蜒曲折的河流，受到河岸的限制较少，可以侧向自由发展。当河床弯曲愈来愈大时，河流的上下河段愈来愈接近，曲流呈“Ω”形，出现狭窄的曲流颈。洪水期，曲流颈可能被冲开，河流不经过曲流而直接进入下一河段，这种现象称为裁弯取直。裁弯取直后，弯曲河道被废弃，形如牛轭，称为牛轭湖。',
 '图2-5\u3000曲流与牛轭湖',
 '图2-6\u3000牛轭湖形成示意',
 '雅鲁藏布大

In [15]:
md_path='./data/cleaned_湘教版地理必修一Chap2.md'
graph_bx1=init_graph(embeddings_model)
graph_bx1=build_knowledge_graph(md_path, llm, embeddings_model,threshold=0.9,k=5)

一共147段
第1段
第2段
第3段
第4段
existed 嘉陵江
existed 青居镇
第5段
existed 嘉陵江
第6段
existed 流水
existed 地表形态
第7段
existed 流水
第8段
第9段
第10段
existed 河漫滩
第11段
第12段
existed 河流阶地
第13段
existed 河床
existed 洪水期
第14段
existed 曲流
existed 牛轭湖
第15段
existed 牛轭湖
existed 河流
existed 洪水期
existed 裁弯取直
第16段
第17段
existed 雅鲁藏布大峡谷
existed 地貌
第18段
existed 雅鲁藏布大峡谷
第19段
第20段
existed 地面
第21段
existed 河流
existed 地形
existed 冲积扇
第22段
existed 冲积扇
第23段
existed 洪积扇
第24段
existed 河流
第25段
existed 尼罗河三角洲
第26段
existed 河流
existed 沙洲
existed 泥沙
第27段
existed 流水侵蚀地貌
existed 流水堆积地貌
第28段
existed 流水侵蚀
第29段
existed 微地貌
existed 流水地貌
第30段
existed 流水侵蚀
existed 山区
第31段
existed 滑坡
existed 河道
第32段
existed 滑坡
第33段
第34段
existed 泥石流
existed 重力
existed 自然灾害
第35段
existed 泥石流
第36段
existed 暴雨
第37段
existed 沟谷
第38段
第39段
existed 泥石流
第40段
第41段
existed 山区
existed 泥石流
existed 泥石流沟
第42段
existed 泥石流
第43段
existed 泥石流
existed 我国
第44段
existed 泥石流
第45段
existed 水
existed 地表
第46段
existed 黄土高原
第47段
existed 黄土高原
existed 地貌
第48段
existed 黄土高原
existed 自然灾害
第49段
第50段
existed 河漫滩
exist

In [79]:
graph_chap2

{'nodes': {'地球': {'aliases': {'地球'}},
  '地貌': {'aliases': {'地形', '地表形态', '地貌', '地面', '峡谷', '阶地'}},
  '流水': {'aliases': {'山洪',
    '曲水',
    '流水',
    '流水下切作用',
    '流水侵蚀',
    '流水侵蚀地貌',
    '流水地貌',
    '流水堆积地貌'}},
  '波浪': {'aliases': {'波浪'}},
  '冰川': {'aliases': {'冰川'}},
  '四川': {'aliases': {'南充', '四川'}},
  '嘉陵江': {'aliases': {'嘉陵江'}},
  '青居镇': {'aliases': {'青居镇'}},
  '码头': {'aliases': {'上码头', '下码头', '码头'}},
  '河道': {'aliases': {'河床', '河流流向', '河谷横断面', '河道'}},
  '航程': {'aliases': {'航程'}},
  '纤夫': {'aliases': {'纤夫'}},
  '客栈': {'aliases': {'客栈'}},
  '遥感影像': {'aliases': {'遥感影像'}},
  '外力作用': {'aliases': {'外力作用'}},
  '湿润山区': {'aliases': {'半湿润山区', '湿润山区'}},
  'V字形': {'aliases': {'V字形'}},
  '深邃峡谷': {'aliases': {'河谷', '深邃峡谷'}},
  '枯水期': {'aliases': {'枯水位', '枯水期', '洪水期'}},
  '河漫滩': {'aliases': {'河流阶地', '河漫滩'}},
  '组成物质': {'aliases': {'组成物质'}},
  '颗粒': {'aliases': {'颗粒'}},
  '土质': {'aliases': {'土质'}},
  '河谷': {'aliases': {'河谷'}},
  '洪水位': {'aliases': {'洪水位'}}},
 'edges': [{'origin': '流水', 'relati

In [16]:
import json

graph_to_save = {
    "nodes": {
        k: {
            "aliases": list(v["aliases"])
        }
        for k, v in graph_bx1["nodes"].items()
    },
    "edges": graph_bx1["edges"]
}

with open("graph_c2.json", "w", encoding="utf-8") as f:
    json.dump(graph_to_save, f, ensure_ascii=False, indent=2)

In [66]:
print(len(graph_bx1['nodes']))
print(len(graph_bx1['edges']))

737
2963


In [65]:
md_path='./data/cleaned_湘教版地理必修一.md'
graph_bx1=build_knowledge_graph(md_path, llm, embeddings_model)

一共584段
第1段
第2段
第3段
第4段
第5段
第6段
第7段
第8段
第9段
第10段
第11段
第12段
第13段
第14段
第15段
第16段
第17段
第18段
第19段
第20段
第21段
第22段
第23段
第24段
第25段
第26段
第27段
第28段
第29段
第30段
第31段
第32段
第33段
第34段
第35段
第36段
第37段
第38段
第39段
第40段
第41段
第42段
第43段
第44段
第45段
第46段
第47段
第48段
第49段
第50段
第51段
第52段
第53段
第54段
第55段
第56段
第57段
第58段
第59段
第60段
第61段
第62段
第63段
第64段
第65段
第66段
第67段
第68段
第69段
第70段
第71段
第72段
第73段
第74段
第75段
第76段
第77段
第78段
第79段
第80段
第81段
第82段
第83段
第84段
第85段
第86段
第87段
第88段
第89段
第90段
第91段
第92段
第93段
第94段
第95段
第96段
第97段
第98段
第99段
第100段
第101段
第102段
第103段
第104段
第105段
第106段
第107段
第108段
第109段
第110段
第111段
第112段
第113段
第114段
第115段
第116段
第117段
第118段
第119段
第120段
第121段
第122段
第123段
第124段
第125段
第126段
第127段
第128段
第129段
第130段
第131段
第132段
第133段
第134段
第135段
第136段
第137段
第138段
第139段
第140段
第141段
第142段
第143段
第144段
第145段
第146段
第147段
第148段
第149段
第150段
第151段
第152段
第153段
第154段
第155段
第156段
第157段
第158段
第159段
第160段
第161段
第162段
第163段
第164段
第165段
第166段
第167段
第168段
第169段
第170段
第171段
第172段
第173段
第174段
第175段
第176段
第177段
第178段
第179段
第180段
第181段
第182段
第183段
第18